In [11]:
# Устанавливаем современную библиотеку для токенизации
!pip install razdel scikit-learn nltk --quiet

import re
import nltk
from nltk.corpus import stopwords
from razdel import tokenize

# Скачиваем стоп-слова
nltk.download('stopwords', quiet=True)
russian_stopwords = stopwords.words('russian')

def clean_text(text):
    # 1. Переводим в нижний регистр
    text = str(text).lower()
    # 2. Убираем ссылки и @упоминания (мусор из Telegram)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)

    # 3. Токенизация с помощью библиотеки razdel (разбиваем на слова без мусора)
    tokens = [word.text for word in tokenize(text) if word.text.isalpha()]

    # 4. Удаляем стоп-слова
    cleaned_words = [word for word in tokens if word not in russian_stopwords]

    return " ".join(cleaned_words)

# Проверяем работу функции на тестовом сообщении
sample_message = "Привет! Зайди на сайт https://google.com и проверь @admin, как тут всё круто работает!"
print("Исходный текст:", sample_message)
print("Очищенный текст:", clean_text(sample_message))

Исходный текст: Привет! Зайди на сайт https://google.com и проверь @admin, как тут всё круто работает!
Очищенный текст: привет зайди сайт проверь всё круто работает


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# 1. Готовим небольшой учебный датасет с примерами (в реальности сюда можно загрузить твой файл с Telegram-сообщениями)
# 0 - безопасное/нейтральное сообщение, 1 - токсичное/нарушающее правила
train_texts = [
    "привет как дела отличный день",
    "спасибо большое за помощь все супер",
    "ужасный сервис полный обман и мошенники",
    "заблокируйте этого идиота он спамит",
    "хорошая погода прекрасное настроение",
    "ненавижу эту контору отвратительное качество"
]
train_labels = [0, 0, 1, 1, 0, 1]

# Предварительно очищаем все тренировочные тексты нашей функцией clean_text
cleaned_train_texts = [clean_text(text) for text in train_texts]

# 2. Создаем пайплайн: перевод текста в TF-IDF векторы + классификатор (Логистическая регрессия)
model = make_pipeline(TfidfVectorizer(), LogisticRegression())

# 3. Обучаем модель на наших данных
model.fit(cleaned_train_texts, train_labels)

# 4. Проверяем работу модели на новом сообщении
new_messages = [
    "ты мне очень помог спасибо",
    "полный отстой верните деньги мошенники"
]

# Очищаем новые сообщения перед предсказанием
cleaned_new_messages = [clean_text(msg) for msg in new_messages]
predictions = model.predict(cleaned_new_messages)

# Выводим результаты
for msg, pred in zip(new_messages, predictions):
    status = "Токсично / Спам 🔴" " (Класс 1)" if pred == 1 else "Безопасно 🟢" " (Класс 0)"
    print(f"Сообщение: '{msg}' --> Вердикт: {status}")

Сообщение: 'ты мне очень помог спасибо' --> Вердикт: Безопасно 🟢 (Класс 0)
Сообщение: 'полный отстой верните деньги мошенники' --> Вердикт: Токсично / Спам 🔴 (Класс 1)


In [13]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import VotingClassifier

# Создаем три разные классические модели без трансформеров
model1 = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression())
model2 = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), MultinomialNB())

# Объединяем их в ансамбль (VotingClassifier с мягким голосованием по вероятностям)
ensemble_model = VotingClassifier(
    estimators=[('lr', model1), ('nb', model2)],
    voting='soft',
    weights=[0.6, 0.4]  # Веса моделей
)

# Обучаем ансамбль
ensemble_model.fit(cleaned_train_texts, train_labels)

# Проверяем ансамбль на тех же сообщениях
ensemble_preds = ensemble_model.predict(cleaned_new_messages)

print("--- Результаты работы ансамбля моделей ---")
for msg, pred in zip(new_messages, ensemble_preds):
    status = "Токсично / Спам 🔴" if pred == 1 else "Безопасно 🟢"
    print(f"Сообщение: '{msg}' --> Вердикт ансамбля: {status}")

--- Результаты работы ансамбля моделей ---
Сообщение: 'ты мне очень помог спасибо' --> Вердикт ансамбля: Безопасно 🟢
Сообщение: 'полный отстой верните деньги мошенники' --> Вердикт ансамбля: Токсично / Спам 🔴
